# 10.2 · A prediction API with FastAPI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/02-fastapi/lesson.ipynb)

Read the [README](README.md) first, and open [api.py](api.py) next to this notebook: it is short. Here we test the service with FastAPI's `TestClient`, which calls the app directly, so nothing needs to be started and nothing is left running.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
import importlib.util

def load_module(rel, name):
    """Import a .py file from the course as a module (so the notebook can test it)."""
    spec = importlib.util.spec_from_file_location(name, repo_file(rel))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

from fastapi.testclient import TestClient
api = load_module("modules/10-deployment-mlops/02-fastapi/api.py", "api")
client = TestClient(api.app)            # sends requests straight to the app: no server, no port

## 1. Is it alive?

`GET /health` is the first thing every monitoring system calls. It should be cheap and say which model is being served.

In [3]:
r = client.get("/health")
print(r.status_code)
r.json()

200


{'status': 'ok',
 'model_trained': '2026-10-28',
 'test_auc': 0.741,
 'threshold': 0.07,
 'features': ['orders', 'spend', 'products', 'tenure_days', 'days_since_last']}

## 2. Score one customer

In [4]:
customer = {"orders": 2, "spend": 310, "products": 14, "tenure_days": 150, "days_since_last": 95}

r = client.post("/predict", json=customer)
print(r.status_code, r.json())

200 {'churn_probability': 0.5244, 'voucher': True, 'threshold': 0.07}


Same answer as the bundle in 10.1 (0.5244). The API adds no maths; it adds a contract and validation.

## 3. Bad input never reaches the model

Pydantic checks every field against the limits in `Customer`. Each problem is reported with its location and a message, and the status is 422.

In [5]:
for bad in [{**customer, "spend": -50},
            {**customer, "orders": "two"},
            {k: v for k, v in customer.items() if k != "products"},
            {**customer, "days_since_last": 200}]:
    r = client.post("/predict", json=bad)
    print(r.status_code, [(e["loc"][-1], e["msg"]) for e in r.json()["detail"]])

422 [('spend', 'Input should be greater than or equal to 0')]
422 [('orders', 'Input should be a valid integer, unable to parse string as an integer')]
422 [('products', 'Field required')]
422 [('body', 'Value error, days_since_last cannot be larger than tenure_days')]


The last one passes every single-field check but fails the rule that the latest order cannot be before the first: `model_validator` in `api.py` catches it.

## 4. Many customers at once

In [6]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
X_train, X_test, y_train, y_test = train_test_split(
    cust[cm.FEATURES], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, train {len(X_train):,} / test {len(X_test):,}")

batch = X_test.astype({"spend": float}).to_dict(orient="records")
r = client.post("/predict/batch", json=batch)
api_p = np.array([row["churn_probability"] for row in r.json()])
print(r.status_code, len(r.json()), "predictions")
print("same as the bundle:", np.allclose(api_p, bundle["pipeline"].predict_proba(X_test)[:, 1], atol=1e-4))

3,317 customers, train 2,487 / test 830
200 830 predictions
same as the bundle: True


## 5. Latency

Time 200 single requests and compute the percentiles from the README. Your numbers will differ: they depend on the machine. The shape (a tight middle, a longer tail) will not.

In [7]:
import time
timings = []
for _ in range(200):
    start = time.perf_counter()
    client.post("/predict", json=customer)
    timings.append((time.perf_counter() - start) * 1000)
t = np.array(timings)
print(f"p50 {np.percentile(t, 50):.2f} ms   p95 {np.percentile(t, 95):.2f} ms   "
      f"mean {t.mean():.2f} ms   max {t.max():.2f} ms")
print(f"one worker, one at a time: about {1000 / t.mean():.0f} requests per second")

p50 1.44 ms   p95 1.71 ms   mean 1.57 ms   max 20.46 ms
one worker, one at a time: about 636 requests per second


In [8]:
start = time.perf_counter()
for row in batch:
    client.post("/predict", json=row)
one_by_one = time.perf_counter() - start
start = time.perf_counter()
client.post("/predict/batch", json=batch)
together = time.perf_counter() - start
print(f"{len(batch)} customers: one at a time {one_by_one:.2f} s, one batch request {together:.3f} s "
      f"({one_by_one / together:.0f}x faster)")

830 customers: one at a time 1.19 s, one batch request 0.004 s (289x faster)


Most of the time per request is not the model: it is turning JSON into Python objects, validating them, building a one-row DataFrame and turning the answer back into JSON. A batch pays those costs once.

## 6. The docs page, as data

FastAPI generates an OpenAPI description of the contract from the code. The `/docs` page you see in a browser is drawn from it, and other teams can generate client code from it.

In [9]:
spec = client.get("/openapi.json").json()
for path, methods in spec["paths"].items():
    for method, info in methods.items():
        print(f"{method.upper():5s} {path:16s} {info.get('summary', '')}")
print("\nCustomer fields:", list(spec["components"]["schemas"]["Customer"]["properties"]))

GET   /health          Health
POST  /predict         Predict
POST  /predict/batch   Predict Batch

Customer fields: ['orders', 'spend', 'products', 'tenure_days', 'days_since_last']


## 7. Running it for real

In a terminal, from this folder, with the course environment active:

```bash
python -m uvicorn api:app --reload
```

Then open http://127.0.0.1:8000/docs, or call it with curl (macOS/Linux) or `Invoke-RestMethod` (Windows PowerShell) as shown in the README. Press Ctrl+C to stop.

## Quiz answers

1. "Unprocessable content": the request's data failed validation. FastAPI produces it automatically from the Pydantic model, before `predict` runs.
2. Loading reads and unpickles a file; doing it per request adds that cost to every call (and risks serving a half-written file during an update).
3. The mean is pulled by a few slow requests and hides them at the same time; p95 tells you what the slowest 1 in 20 users experience.
4. GET asks for information and sends no body; POST sends data (the customer) for the server to process.
5. Far less overhead (one validation, one DataFrame, one model call) and far fewer network round trips; also simpler for the caller.

**Next:** [10.3 A web app with Streamlit](../03-streamlit/)